# SL-7 - LLMs et Apprentissage Symbolique : Generation et Verification d'Hypotheses

**Navigation** : [Index](./README.md) | [<< Knowledge Graphs ILP](SL-6-KnowledgeGraphs-ILP.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Utiliser un LLM comme **generateur d'hypotheses** a partir d'exemples structures
2. Verifier formellement les regles generees par un LLM avec un **oracle symbolique**
3. Comprendre le parallele entre **Chain-of-Thought** et l'apprentissage base sur les explications (EBL)
4. Implementer un pipeline complet : exemples -> LLM -> candidats -> verification -> regles validees
5. Brancher un **vrai LLM** (OpenRouter, OpenAI ou modele auto-heberge) via un fichier `.env`
6. Evaluer les forces et limites de l'approche neuro-symbolique pour l'apprentissage de regles

### Prerequis
- SL-1 a SL-4 : Apprentissage logique, EBL/RBL, ILP
- Python 3.10+
- Notions de base sur les LLMs (prompting, generation de texte)

### Duree estimee : 40 minutes

### References
- Russell & Norvig, *Artificial Intelligence: A Modern Approach*, 4e ed., Chapitre 19
- Wei et al., *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* (2022)
- Muggleton, *Inductive Logic Programming* (1991)
- Pan et al., *Unifying Large Language Models and Knowledge Graphs: A Roadmap* (2024)

In [1]:
# Imports et configuration
from typing import Optional
from dataclasses import dataclass, field
from collections import defaultdict
import itertools
import json
import re

# Metadata du notebook
NOTEBOOK_INFO = {
    "title": "SL-7 - LLMs et Apprentissage Symbolique",
    "series": "SymbolicLearning",
    "aima_chapters": ["19 (extension)"],
    "date": "2026-05",
}

print(f"Notebook : {NOTEBOOK_INFO['title']}")
print(f"Chapitres AIMA : {', '.join(NOTEBOOK_INFO['aima_chapters'])}")
print("Mode : sections 1-6 = simulateur local deterministe (aucune API requise)")
print("       section 7  = integration d'un vrai LLM via .env (optionnelle)")

Notebook : SL-7 - LLMs et Apprentissage Symbolique
Chapitres AIMA : 19 (extension)
Mode : sections 1-6 = simulateur local deterministe (aucune API requise)
       section 7  = integration d'un vrai LLM via .env (optionnelle)


---

## 1. Introduction : LLMs comme moteurs de raisonnement symbolique

Les grands modeles de langage (LLMs) comme GPT-4, Claude ou Llama sont capables de generer du texte structure, y compris des **regles logiques** et des **explications de raisonnement**. Cette capacite ouvre une nouvelle voie pour l'apprentissage symbolique :

### Le paradigme neuro-symbolique pour l'apprentissage

| Etape | Role | Analogue classique |
|-------|------|--------------------|
| Generation d'hypotheses | **LLM** (neural) | Exploration heuristique |
| Verification formelle | **Oracle symbolique** | Test de consistance |
| Selection et raffinement | **Pipeline iteratif** | CBH / Version Space |

### Pourquoi combiner LLMs et symbolique ?

1. **LLMs sont creatifs mais peu fiables** : ils generent des hypotheses plausibles mais peuvent halluciner
2. **Les systemes symboliques sont fiables mais peu creatifs** : ils verifient parfaitement mais n'explorent pas
3. **La combinaison** tire parti des deux : creativite neuronale + rigueur symbolique

### Lien avec AIMA Chapitre 19

Dans les notebooks precedents, nous avons etudie comment un agent peut apprendre des regles a partir d'exemples (SL-1), de connaissances de fond (SL-2), ou par programmation logique inductive (SL-4). Ce notebook montre comment les LLMs peuvent servir de **generateur d'hypotheses** dans ce cadre, en remplacement ou en complement des methodes de recherche classique.

> **Note** : Ce notebook fonctionne **entierement en local** sans cle API. Les reponses LLM sont simulees par des templates pre-definis qui illustrent le comportement attendu. Les marqueurs `# TODO` indiquent ou brancher une vraie API.

In [2]:
# Le paradigme neuro-symbolique, quantifie : pourquoi deleguer la GENERATION ?
# Cout de l'approche classique = explorer un espace d'hypotheses qui explose
# combinatoirement ; cout de l'approche LLM = quelques candidats cibles a verifier.

from math import prod


def taille_espace(domaines: list[int]) -> int:
    """Taille de l'espace des hypotheses conjonctives (cf. SL-1).

    Chaque attribut prend une valeur specifique ou le joker '?',
    plus l'unique hypothese vide qui rejette tout.
    """
    return prod(d + 1 for d in domaines) + 1


scenarios = [
    ("Restaurant SL-1 (10 attributs)", [2, 2, 2, 2, 3, 3, 2, 2, 4, 4]),
    ("20 attributs binaires", [2] * 20),
    ("30 attributs binaires", [2] * 30),
]

K_LLM = 5  # un LLM propose typiquement quelques hypotheses ciblees par prompt

print("Paradigme neuro-symbolique : generation ciblee vs enumeration")
print("=" * 66)
print()
print(f"{'Domaine':32s} | {'|H| (enumeration)':>18s} | {'Candidats LLM':>13s}")
print("-" * 70)
for nom, domaines in scenarios:
    print(f"{nom:32s} | {taille_espace(domaines):18,d} | {K_LLM:13d}")

print()
print("Verifier la consistance d'UN candidat reste rapide (lineaire dans les")
print("exemples) : c'est l'ENUMERATION de H qui devient intenable. D'ou :")
print()
print("  Approche classique (SL-1 a SL-4) :")
print("    Exemples -> recherche exhaustive/heuristique dans H -> hypotheses")
print("  Approche LLM (ce notebook) :")
print("    Exemples -> LLM (genere ~5 candidats) -> verif symbolique -> regles")
print("    Avantage : creativite, contexte naturel ; risque : hallucinations")
print("  Approche hybride = generation neuronale + filtre symbolique :")
print("    le meilleur des deux mondes")


Paradigme neuro-symbolique : generation ciblee vs enumeration

Domaine                          |  |H| (enumeration) | Candidats LLM
----------------------------------------------------------------------
Restaurant SL-1 (10 attributs)   |            291,601 |             5
20 attributs binaires            |      3,486,784,402 |             5
30 attributs binaires            | 205,891,132,094,650 |             5

Verifier la consistance d'UN candidat reste rapide (lineaire dans les
exemples) : c'est l'ENUMERATION de H qui devient intenable. D'ou :

  Approche classique (SL-1 a SL-4) :
    Exemples -> recherche exhaustive/heuristique dans H -> hypotheses
  Approche LLM (ce notebook) :
    Exemples -> LLM (genere ~5 candidats) -> verif symbolique -> regles
    Avantage : creativite, contexte naturel ; risque : hallucinations
  Approche hybride = generation neuronale + filtre symbolique :
    le meilleur des deux mondes


### Interpretation : Le paradigme neuro-symbolique

| Approche | Creativite | Fiabilite | Vitesse | Domaine couvert |
|----------|-----------|-----------|---------|----------------|
| Classique (CBH/VS/ILP) | Faible | Elevee | Lente | Restreint par la syntaxe |
| LLM seul | Elevee | Faible | Rapide | Tres large |
| **Hybride** | **Elevee** | **Elevee** | **Moyenne** | **Large** |

> **Point cle** : Le LLM genere des hypotheses plausibles en langage naturel, puis le systeme symbolique filtre les candidats incorrects. C'est une forme de **generate-and-test** a grande echelle.

---

## 2. LLM comme generateur d'hypotheses

La premiere etape du pipeline est d'utiliser un LLM pour generer des **regles logiques** (clauses de Horn) a partir d'exemples positifs et negatifs.

### Principe

1. Presenter les exemples au LLM sous forme structuree (tableau, JSON)
2. Demander au LLM de proposer des regles qui expliquent les exemples positifs et excluent les negatifs
3. Collecter les regles generees comme **candidats** a verifier

### Format des regles : clauses de Horn

Une clause de Horn est une implication de la forme :

$$
P_1 \wedge P_2 \wedge \ldots \wedge P_n \Rightarrow Q
$$

ou les $P_i$ sont des litteraux (conditions) et $Q$ est le predicat cible.

In [3]:
# Domaine du restaurant (reutilise de SL-1)
ATTRIBUTES = [
    "Alternate", "Bar", "Fri/Sat", "Hungry",
    "Patrons", "Price", "Raining", "Reservation",
    "Type", "WaitEstimate"
]

# Les 12 exemples du restaurant, adaptes de la Table 19.1 d'AIMA.
# Comme en SL-1, les labels de X2 et X3 sont inverses par rapport au livre :
# ainsi AUCUNE clause de Horn unique n'est consistante (meme Patrons=Some a un
# contre-exemple), ce qui est exactement le scenario que l'oracle doit detecter.
RAW_EXAMPLES = [
    ("Yes", "No",  "No",  "Yes", "Some", "$$$", "No",  "Yes",  "French", "0-10",  True),
    ("Yes", "No",  "No",  "Yes", "Full", "$",   "No",  "No",   "Thai",   "30-60", True),
    ("No",  "Yes", "No",  "No",  "Some", "$",   "No",  "No",   "Burger", "0-10",  False),
    ("Yes", "No",  "Yes", "Yes", "Full", "$",   "Yes", "No",   "Thai",   "10-30", True),
    ("Yes", "No",  "Yes", "No",  "Full", "$$$", "No",  "Yes",  "French", ">60",   False),
    ("No",  "Yes", "No",  "Yes", "Some", "$$",  "Yes", "Yes",  "Italian","0-10",  True),
    ("No",  "Yes", "No",  "No",  "None", "$",   "Yes", "No",  "Burger", "0-10",  False),
    ("No",  "No",  "No",  "Yes", "Some", "$$",  "Yes", "Yes",  "Thai",   "0-10",  True),
    ("No",  "Yes", "Yes", "No",  "Full", "$",   "Yes", "No",  "Burger", "10-30", False),
    ("Yes", "Yes", "Yes", "Yes", "Full", "$$$", "No",  "Yes",  "Italian","10-30", False),
    ("No",  "No",  "No",  "No",  "None", "$",   "No",  "No",   "Thai",   "0-10",  False),
    ("Yes", "Yes", "Yes", "Yes", "Full", "$",   "No",  "No",   "Burger", "30-60", True),
]

def parse_example(raw: tuple) -> dict:
    """Convertit un tuple brut en dictionnaire avec attributs + label."""
    attrs = {ATTRIBUTES[i]: raw[i] for i in range(len(ATTRIBUTES))}
    attrs["WillWait"] = raw[len(ATTRIBUTES)]
    return attrs

EXAMPLES = [parse_example(ex) for ex in RAW_EXAMPLES]
POSITIVES = [e for e in EXAMPLES if e["WillWait"]]
NEGATIVES = [e for e in EXAMPLES if not e["WillWait"]]

print(f"Domaine : {len(ATTRIBUTES)} attributs")
print(f"Exemples : {len(EXAMPLES)} ({len(POSITIVES)} positifs, {len(NEGATIVES)} negatifs)")
print()
print("Exemples positifs :")
for e in POSITIVES:
    print(f"  Patrons={e['Patrons']:5s}  Hungry={e['Hungry']:3s}  Fri/Sat={e['Fri/Sat']:3s}  Type={e['Type']:8s}")

Domaine : 10 attributs
Exemples : 12 (6 positifs, 6 negatifs)

Exemples positifs :
  Patrons=Some   Hungry=Yes  Fri/Sat=No   Type=French  
  Patrons=Full   Hungry=Yes  Fri/Sat=No   Type=Thai    
  Patrons=Full   Hungry=Yes  Fri/Sat=Yes  Type=Thai    
  Patrons=Some   Hungry=Yes  Fri/Sat=No   Type=Italian 
  Patrons=Some   Hungry=Yes  Fri/Sat=No   Type=Thai    
  Patrons=Full   Hungry=Yes  Fri/Sat=Yes  Type=Burger  


### Construction du prompt de generation de regles

Le domaine etant defini avec ses 12 exemples et 10 attributs, l'etape suivante consiste a construire un **prompt structure** qui sera soumis au LLM pour qu'il genere des hypotheses. Ce prompt suit un format systematique :

1. **Role** : "Tu es un expert en apprentissage symbolique" (guidage du comportement)
2. **Contexte** : les attributs disponibles et la cible (`WillWait`)
3. **Exemples** : les cas positifs et negatifs presentes ligne par ligne
4. **Format attendu** : clauses de Horn (`condition1 AND condition2 => WillWait`)

En pratique avec un vrai LLM, la qualite du prompt determine directement la qualite des hypotheses generees.

In [4]:
# Construction du prompt pour le LLM (simule)

def build_rule_prompt(examples: list[dict], target: str = "WillWait") -> str:
    """Construit un prompt structure pour demander des regles au LLM."""
    lines = [
        "Tu es un expert en apprentissage symbolique.",
        "Genere des regles logiques (clauses de Horn) qui expliquent les donnees suivantes.",
        "",
        f"Attributs disponibles : {', '.join(ATTRIBUTES)}",
        f"Cible : {target} (True/False)",
        "",
        "Exemples positifs (WillWait=True) :",
    ]
    
    for i, e in enumerate(POSITIVES):
        attrs_str = ", ".join(f"{a}={e[a]}" for a in ATTRIBUTES)
        lines.append(f"  {i+1}. {attrs_str}")
    
    lines.append("")
    lines.append("Exemples negatifs (WillWait=False) :")
    for i, e in enumerate(NEGATIVES):
        attrs_str = ", ".join(f"{a}={e[a]}" for a in ATTRIBUTES)
        lines.append(f"  {i+1}. {attrs_str}")
    
    lines.extend([
        "",
        "Format de reponse attendu (une regle par ligne) :",
        "  condition1 AND condition2 => WillWait",
        "  condition3 => NOT WillWait",
        "",
        "Genere 3 a 5 regles candidates.",
    ])
    
    return "\n".join(lines)


prompt = build_rule_prompt(EXAMPLES)
print("Prompt envoye au LLM (simule) :")
print("=" * 60)
print(prompt)

Prompt envoye au LLM (simule) :
Tu es un expert en apprentissage symbolique.
Genere des regles logiques (clauses de Horn) qui expliquent les donnees suivantes.

Attributs disponibles : Alternate, Bar, Fri/Sat, Hungry, Patrons, Price, Raining, Reservation, Type, WaitEstimate
Cible : WillWait (True/False)

Exemples positifs (WillWait=True) :
  1. Alternate=Yes, Bar=No, Fri/Sat=No, Hungry=Yes, Patrons=Some, Price=$$$, Raining=No, Reservation=Yes, Type=French, WaitEstimate=0-10
  2. Alternate=Yes, Bar=No, Fri/Sat=No, Hungry=Yes, Patrons=Full, Price=$, Raining=No, Reservation=No, Type=Thai, WaitEstimate=30-60
  3. Alternate=Yes, Bar=No, Fri/Sat=Yes, Hungry=Yes, Patrons=Full, Price=$, Raining=Yes, Reservation=No, Type=Thai, WaitEstimate=10-30
  4. Alternate=No, Bar=Yes, Fri/Sat=No, Hungry=Yes, Patrons=Some, Price=$$, Raining=Yes, Reservation=Yes, Type=Italian, WaitEstimate=0-10
  5. Alternate=No, Bar=No, Fri/Sat=No, Hungry=Yes, Patrons=Some, Price=$$, Raining=Yes, Reservation=Yes, Type=Thai,

### Simulateur de LLM

Pour que ce notebook fonctionne sans cle API, nous simulons les reponses du LLM avec des **templates pre-definis**. Ces templates representent les reponses typiques d'un LLM competent face a ce type de prompt.

> **Note** : En production, remplacer `simulate_llm_response()` par un appel a l'API OpenAI, Anthropic ou un modele local.

In [5]:
# Simulateur de reponses LLM

@dataclass
class HornClause:
    """Represente une clause de Horn : conditions => conclusion."""
    conditions: list[str]   # Liste des conditions (litteraux)
    conclusion: str         # Predicat cible (ex: "WillWait")
    negated: bool = False   # Si True, la conclusion est niee
    
    def __str__(self) -> str:
        cond_str = " AND ".join(self.conditions)
        neg = "NOT " if self.negated else ""
        return f"{cond_str} => {neg}{self.conclusion}"
    
    def evaluate(self, example: dict) -> Optional[bool]:
        """Evalue la clause sur un exemple.
        
        Retourne :
        - True si toutes les conditions sont satisfaites (la regle s'applique)
        - False si au moins une condition n'est pas satisfaite (la regle ne s'applique pas)
        """
        for cond in self.conditions:
            if "=" in cond:
                attr, val = cond.split("=", 1)
                attr = attr.strip()
                val = val.strip()
                if attr in example and example[attr] != val:
                    return False
        return True


def simulate_llm_response(prompt: str) -> list[HornClause]:
    """Simule la reponse d'un LLM au prompt de generation de regles.
    
    Pour brancher un vrai LLM, voir la section 7 (integration reelle via
    .env) ; references :
      - OpenAI : https://platform.openai.com/docs/api-reference/chat
      - Anthropic : https://docs.anthropic.com/en/docs/about-claude/models
      - Ollama (local) : https://github.com/ollama/ollama
    
    Exemple avec l'API OpenAI :
        import openai
        response = openai.chat.completions.create(
            model="gpt-4", messages=[{"role": "user", "content": prompt}]
        )
        return parse_llm_rules(response.choices[0].message.content)
    """
    # Reponses simulees representant des hypotheses plausibles
    # Un vrai LLM genererait des regles similaires (mais potentiellement differentes)
    return [
        HornClause(
            conditions=["Patrons=Some"],
            conclusion="WillWait"
        ),
        HornClause(
            conditions=["Patrons=Full", "Hungry=Yes"],
            conclusion="WillWait"
        ),
        HornClause(
            conditions=["Hungry=Yes"],
            conclusion="WillWait"
        ),
        HornClause(
            conditions=["Patrons=None"],
            conclusion="WillWait",
            negated=True
        ),
        HornClause(
            conditions=["Patrons=Full", "Price=$$$"],
            conclusion="WillWait",
            negated=True
        ),
    ]


# Generation des hypotheses
candidate_rules = simulate_llm_response(prompt)

print("Regles candidates generees par le LLM (simule) :")
print("=" * 60)
for i, rule in enumerate(candidate_rules):
    print(f"  R{i+1}: {rule}")
print(f"\nTotal : {len(candidate_rules)} regles candidates")

Regles candidates generees par le LLM (simule) :
  R1: Patrons=Some => WillWait
  R2: Patrons=Full AND Hungry=Yes => WillWait
  R3: Hungry=Yes => WillWait
  R4: Patrons=None => NOT WillWait
  R5: Patrons=Full AND Price=$$$ => NOT WillWait

Total : 5 regles candidates


### Interpretation : Generation d'hypotheses par LLM

Le LLM a genere 5 regles candidates :

| Regle | Conditions | Conclusion | Type |
|-------|-----------|------------|------|
| R1 | Patrons=Some | WillWait | Regle positive simple |
| R2 | Patrons=Full AND Hungry=Yes | WillWait | Regle positive conjonctive |
| R3 | Hungry=Yes | WillWait | Regle positive (trop generale ?) |
| R4 | Patrons=None | NOT WillWait | Regle negative |
| R5 | Patrons=Full AND Price=$$$ | NOT WillWait | Regle negative conjonctive |

**Observations** :
- R1 et R2 visent la disjonction classique du domaine restaurant (`Patrons=Some OU (Patrons=Full AND Hungry=Yes)`) -- mais on verra que, dans notre variante des donnees, meme elles ont chacune un contre-exemple
- R3 est une **sur-generalisation** (Hungry=Yes couvre aussi des negatifs)
- R4 et R5 sont des regles negatives complementaires

> **Point cle** : Le LLM genere des hypotheses plausibles mais certaines sont incorrectes. L'etape suivante (verification symbolique) est indispensable.

---

## 3. Verification symbolique des sorties LLM

La deuxieme etape du pipeline est de **verifier formellement** chaque regle candidate contre les donnees d'entrainement. C'est le role de l'**oracle symbolique**.

### Principes de verification

Pour chaque regle candidate, on verifie :
1. **Completude positive** : la regle couvre-t-elle tous les exemples positifs ?
2. **Correction negative** : la regle exclut-elle tous les exemples negatifs ?
3. **Consistance** : la regle est-elle libre de contradictions internes ?
4. **Parsimonie** : la regle est-elle minimale (pas de conditions redondantes) ?

In [6]:
# Oracle symbolique : verification des regles candidates

@dataclass
class VerificationResult:
    """Resultat de la verification d'une regle candidate."""
    rule_index: int
    rule_str: str
    tp: int  # True Positifs : couverts et positifs
    fp: int  # Faux Positifs : couverts mais negatifs
    fn: int  # Faux Negatifs : non couverts mais positifs
    tn: int  # True Negatifs : non couverts et negatifs
    is_consistent: bool
    covers_all_positives: bool
    
    @property
    def precision(self) -> float:
        """Precision = TP / (TP + FP)."""
        total = self.tp + self.fp
        return self.tp / total if total > 0 else 0.0
    
    @property
    def recall(self) -> float:
        """Recall = TP / (TP + FN)."""
        total = self.tp + self.fn
        return self.tp / total if total > 0 else 0.0
    
    @property
    def f1(self) -> float:
        """F1-score."""
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def verify_rule(rule: HornClause, examples: list[dict], rule_idx: int) -> VerificationResult:
    """Verifie une regle candidate contre les exemples."""
    tp = fp = fn = tn = 0
    
    for ex in examples:
        rule_applies = rule.evaluate(ex)
        is_positive = ex["WillWait"]
        
        # Pour les regles niees, inverser la logique
        if rule.negated:
            if rule_applies and not is_positive:
                tp += 1  # Correctement predit negatif
            elif rule_applies and is_positive:
                fp += 1  # Fauxement predit negatif
            elif not rule_applies and not is_positive:
                fn += 1  # Negatif non detecte
            else:
                tn += 1
        else:
            if rule_applies and is_positive:
                tp += 1  # Correctement predit positif
            elif rule_applies and not is_positive:
                fp += 1  # Fauxement predit positif
            elif not rule_applies and is_positive:
                fn += 1  # Positif non couvert
            else:
                tn += 1
    
    is_consistent = (fp == 0)
    covers_all = (fn == 0) if not rule.negated else True
    
    return VerificationResult(
        rule_index=rule_idx,
        rule_str=str(rule),
        tp=tp, fp=fp, fn=fn, tn=tn,
        is_consistent=is_consistent,
        covers_all_positives=covers_all
    )


# Verification de toutes les regles candidates
print("Verification symbolique des regles candidates")
print("=" * 70)
print()
print(f"{'Regle':40s} | {'TP':>3} | {'FP':>3} | {'FN':>3} | {'Prec':>5} | {'Recall':>6} | {'OK':>2}")
print("-" * 80)

results = []
for i, rule in enumerate(candidate_rules):
    vr = verify_rule(rule, EXAMPLES, i)
    results.append(vr)
    ok_str = "OUI" if vr.is_consistent else "NON"
    print(f"R{i+1}: {vr.rule_str:36s} | {vr.tp:>3} | {vr.fp:>3} | {vr.fn:>3} | {vr.precision:>5.2f} | {vr.recall:>6.2f} | {ok_str:>3}")

print()
print("Legende : TP=Vrais Positifs, FP=Faux Positifs, FN=Faux Negatifs, OK=Consistante")

Verification symbolique des regles candidates

Regle                                    |  TP |  FP |  FN |  Prec | Recall | OK
--------------------------------------------------------------------------------
R1: Patrons=Some => WillWait             |   3 |   1 |   3 |  0.75 |   0.50 | NON
R2: Patrons=Full AND Hungry=Yes => WillWait |   3 |   1 |   3 |  0.75 |   0.50 | NON
R3: Hungry=Yes => WillWait               |   6 |   1 |   0 |  0.86 |   1.00 | NON
R4: Patrons=None => NOT WillWait         |   2 |   0 |   4 |  1.00 |   0.33 | OUI
R5: Patrons=Full AND Price=$$$ => NOT WillWait |   2 |   0 |   4 |  1.00 |   0.33 | OUI

Legende : TP=Vrais Positifs, FP=Faux Positifs, FN=Faux Negatifs, OK=Consistante


### Interpretation : Verification symbolique

L'oracle symbolique a identifie les regles correctes et incorrectes :

| Regle | Verdict | TP | FP | Raison |
|-------|---------|----|----|--------|
| R1 (Patrons=Some) | **Inconsistante** | 3 | 1 | 1 faux positif (ex2: Patrons=Some, Hungry=No) |
| R2 (Patrons=Full AND Hungry=Yes) | **Inconsistante** | 3 | 1 | 1 faux positif (ex9: Full, Hungry=Yes, Price=$$$) |
| R3 (Hungry=Yes) | **Inconsistante** | 6 | 1 | Couvre presque tout mais 1 FP |
| R4 (Patrons=None => NOT WillWait) | **Consistante** | 2 | 0 | Regle negative correcte |
| R5 (Patrons=Full AND Price=$$$ => NOT) | **Consistante** | 2 | 0 | Regle negative correcte |

**Observation importante** : Aucune regle positive n'est parfaitement consistante !
C'est voulu : les donnees (labels de X2/X3 inverses par rapport au livre) rendent le
concept strictement **disjonctif avec exceptions** -- il ne peut pas etre capture par
une seule clause de Horn.
L'exemple 9 (Full, Hungry=Yes, Italian, Price=$$$ => WillWait=False) contredit meme R2.

> **Point cle** : L'oracle symbolique detecte que le concept est disjonctif. Seules les regles negatives (R4, R5) passent la verification. C'est une limitation fondamentale de la representation par clauses de Horn individuelles, que nous avions deja rencontree en SL-1 avec le Version Space vide.


In [7]:
# Extraction des regles validees et synthese

validated_rules = []
rejected_rules = []

for i, (rule, vr) in enumerate(zip(candidate_rules, results)):
    if vr.is_consistent:
        validated_rules.append(rule)
    else:
        rejected_rules.append((rule, vr))

print(f"Synthese de la verification :")
print(f"  Regles candidates : {len(candidate_rules)}")
print(f"  Regles validees   : {len(validated_rules)}")
print(f"  Regles rejetees   : {len(rejected_rules)}")
print()

print("Regles validees (prises pour le modele final) :")
for i, rule in enumerate(validated_rules):
    print(f"  V{i+1}: {rule}")

print()
print("Regles rejetees (par l'oracle symbolique) :")
for rule, vr in rejected_rules:
    print(f"  X  : {rule}")
    print(f"       Raison : {vr.fp} faux positifs, precision={vr.precision:.2f}")

# Analyse : pourquoi les regles positives echouent-elles ?
print()
print("Analyse : Le concept WillWait est DISJONCTIF.")
print("Aucune clause de Horn individuelle ne couvre tous les positifs sans FP.")
print("C'est une limitation rencontree en SL-1 (Version Space vide).")
print()
print("Pour couvrir completement le concept, il faut COMBINER plusieurs clauses :")
print("  Patrons=Some => WillWait   (3 TP, 1 FP)")
print("  OU Patrons=Full AND Hungry=Yes => WillWait  (3 TP, 1 FP)")
print("Le pipeline iteratif (section 6) tentera cette combinaison.")


Synthese de la verification :
  Regles candidates : 5
  Regles validees   : 2
  Regles rejetees   : 3

Regles validees (prises pour le modele final) :
  V1: Patrons=None => NOT WillWait
  V2: Patrons=Full AND Price=$$$ => NOT WillWait

Regles rejetees (par l'oracle symbolique) :
  X  : Patrons=Some => WillWait
       Raison : 1 faux positifs, precision=0.75
  X  : Patrons=Full AND Hungry=Yes => WillWait
       Raison : 1 faux positifs, precision=0.75
  X  : Hungry=Yes => WillWait
       Raison : 1 faux positifs, precision=0.86

Analyse : Le concept WillWait est DISJONCTIF.
Aucune clause de Horn individuelle ne couvre tous les positifs sans FP.
C'est une limitation rencontree en SL-1 (Version Space vide).

Pour couvrir completement le concept, il faut COMBINER plusieurs clauses :
  Patrons=Some => WillWait   (3 TP, 1 FP)
  OU Patrons=Full AND Hungry=Yes => WillWait  (3 TP, 1 FP)
Le pipeline iteratif (section 6) tentera cette combinaison.


---

## 4. Chain-of-Thought comme explication

Le **Chain-of-Thought (CoT)** prompting est une technique qui demande au LLM de raisonner etape par etape avant de donner sa reponse. Ce processus presente des parallels profonds avec l'**apprentissage base sur les explications (EBL)** etudie en SL-2.

### Parallele CoT / EBL

| Aspect | Chain-of-Thought (LLM) | EBL (symbolique) |
|--------|----------------------|------------------|
| Entree | Question + contexte | Exemple + theorie de fond |
| Processus | Etapes de raisonnement en langage naturel | Arbre de preuve formel |
| Sortie | Reponse + explication | Regle generalisee |
| Fiabilite | Variable (hallucinations) | Garantie (deduction) |
| Generalite | Contextuelle | Formelle |

### Extraire des "arbres de preuve" des traces CoT

L'idee est de :
1. Demander au LLM de raisonner sur un exemple (CoT)
2. Parser les etapes du raisonnement comme des etapes de preuve
3. Extraire une structure logique de la trace

In [8]:
# Simulation de Chain-of-Thought pour la classification du restaurant

@dataclass
class CoTStep:
    """Une etape dans un raisonnement Chain-of-Thought."""
    step: int
    observation: str
    reasoning: str
    conclusion: str


@dataclass
class CoTTrace:
    """Trace complete d'un raisonnement Chain-of-Thought."""
    example_idx: int
    example_desc: str
    steps: list[CoTStep]
    final_answer: bool
    extracted_rule: Optional[HornClause] = None


def simulate_cot_classification(example: dict, idx: int) -> CoTTrace:
    """Simule un raisonnement Chain-of-Thought pour un exemple du restaurant.
    
    Pour brancher un vrai LLM avec prompting CoT :
        prompt = f"Reason step by step about this restaurant: {example}"
        response = llm_client.chat(prompt)
    """
    steps = []
    
    # Etape 1 : Observer la frequentation
    steps.append(CoTStep(
        step=1,
        observation=f"Patrons = {example['Patrons']}",
        reasoning=f"Le restaurant a une frequentation '{example['Patrons']}'",
        conclusion=f"Patrons est {example['Patrons']}"
    ))
    
    # Etape 2 : Si Full, verifier la faim
    if example["Patrons"] == "Full":
        steps.append(CoTStep(
            step=2,
            observation=f"Hungry = {example['Hungry']}",
            reasoning="Quand le restaurant est plein, la decision depend de la faim",
            conclusion=f"Hungry est {example['Hungry']}"
        ))
    
    # Decision
    if example["Patrons"] == "Some":
        answer = True
        rule = HornClause(["Patrons=Some"], "WillWait")
    elif example["Patrons"] == "Full" and example["Hungry"] == "Yes":
        answer = True
        rule = HornClause(["Patrons=Full", "Hungry=Yes"], "WillWait")
    elif example["Patrons"] == "Full" and example["Hungry"] == "No":
        answer = False
        rule = HornClause(["Patrons=Full", "Hungry=No"], "WillWait", negated=True)
    else:  # Patrons=None
        answer = False
        rule = HornClause(["Patrons=None"], "WillWait", negated=True)
    
    return CoTTrace(
        example_idx=idx,
        example_desc=f"Pat={example['Patrons']}, Type={example['Type']}, Hungry={example['Hungry']}",
        steps=steps,
        final_answer=answer,
        extracted_rule=rule
    )


# Generation de traces CoT pour quelques exemples
test_indices = [0, 1, 2, 4, 6, 11]  # Mix positifs/negatifs

print("Traces Chain-of-Thought (simulees)")
print("=" * 65)

cot_traces = []
for idx in test_indices:
    ex = EXAMPLES[idx]
    trace = simulate_cot_classification(ex, idx)
    cot_traces.append(trace)
    
    label = "+" if ex["WillWait"] else "-"
    print(f"\nEx {idx} [{label}] {trace.example_desc}")
    for step in trace.steps:
        print(f"  Etape {step.step}: {step.observation}")
        print(f"    Raisonnement : {step.reasoning}")
        print(f"    Conclusion   : {step.conclusion}")
    print(f"  => Decision : WillWait={trace.final_answer}")
    if trace.extracted_rule:
        print(f"  => Regle extraite : {trace.extracted_rule}")

Traces Chain-of-Thought (simulees)

Ex 0 [+] Pat=Some, Type=French, Hungry=Yes
  Etape 1: Patrons = Some
    Raisonnement : Le restaurant a une frequentation 'Some'
    Conclusion   : Patrons est Some
  => Decision : WillWait=True
  => Regle extraite : Patrons=Some => WillWait

Ex 1 [+] Pat=Full, Type=Thai, Hungry=Yes
  Etape 1: Patrons = Full
    Raisonnement : Le restaurant a une frequentation 'Full'
    Conclusion   : Patrons est Full
  Etape 2: Hungry = Yes
    Raisonnement : Quand le restaurant est plein, la decision depend de la faim
    Conclusion   : Hungry est Yes
  => Decision : WillWait=True
  => Regle extraite : Patrons=Full AND Hungry=Yes => WillWait

Ex 2 [-] Pat=Some, Type=Burger, Hungry=No
  Etape 1: Patrons = Some
    Raisonnement : Le restaurant a une frequentation 'Some'
    Conclusion   : Patrons est Some
  => Decision : WillWait=True
  => Regle extraite : Patrons=Some => WillWait

Ex 4 [-] Pat=Full, Type=French, Hungry=No
  Etape 1: Patrons = Full
    Raisonnement 

### Interpretation : Chain-of-Thought et EBL

Chaque trace CoT est structuree comme un **arbre de preuve EBL** :

| Element CoT | Equivalent EBL | Description |
|-------------|---------------|-------------|
| Observation initiale | Fait de l'exemple | Point de depart du raisonnement |
| Etape de raisonnement | Application d'une regle | Inference logique |
| Conclusion intermediaire | Fait derive | Resultat d'une etape |
| Decision finale | Conclusion de la preuve | Predicat cible |
| Regle extraite | Regle EBL | Generalisation de la trace |

> **Attention (Ex 2)** : la trace simulee conclut `WillWait=True` sur l'exemple 2
> alors que son label est **False** -- le "raisonnement" applique l'heuristique
> `Patrons=Some` sans verifier le reste. Une chaine de raisonnement plausible peut
> donc etre fausse : c'est precisement pourquoi la regle extraite repasse par
> l'oracle symbolique (qui la rejettera, section 5).

> **Point cle** : Le CoT est une forme "approximative" d'arbre de preuve. La difference principale est que le CoT peut contenir des erreurs logiques, tandis que l'arbre de preuve EBL est formellement correct. La verification symbolique permet de detecter ces erreurs.

In [9]:
# Extraction de regles a partir des traces CoT

print("Extraction de regles a partir des traces CoT")
print("=" * 55)
print()

extracted_rules = []
for trace in cot_traces:
    if trace.extracted_rule:
        extracted_rules.append(trace.extracted_rule)

# Deduplication
unique_rules = []
seen = set()
for rule in extracted_rules:
    rule_key = str(rule)
    if rule_key not in seen:
        seen.add(rule_key)
        unique_rules.append(rule)

print(f"Regles extraites des {len(cot_traces)} traces CoT :")
print(f"  Total (avant deduplication) : {len(extracted_rules)}")
print(f"  Uniques : {len(unique_rules)}")
print()

for i, rule in enumerate(unique_rules):
    print(f"  C{i+1}: {rule}")

print()
print("Verification des regles extraites (vs regles du LLM) :")
print(f"  Regles LLM direct   : {len(candidate_rules)} candidates")
print(f"  Regles CoT extraites : {len(unique_rules)} uniques")
print()
print("Les regles extraites des traces CoT sont plus ciblees")
print("car elles reflettent le raisonnement effectif sur chaque exemple,")
print("tandis que les regles directes sont des hypotheses globales.")


Extraction de regles a partir des traces CoT

Regles extraites des 6 traces CoT :
  Total (avant deduplication) : 6
  Uniques : 4

  C1: Patrons=Some => WillWait
  C2: Patrons=Full AND Hungry=Yes => WillWait
  C3: Patrons=Full AND Hungry=No => NOT WillWait
  C4: Patrons=None => NOT WillWait

Verification des regles extraites (vs regles du LLM) :
  Regles LLM direct   : 5 candidates
  Regles CoT extraites : 4 uniques

Les regles extraites des traces CoT sont plus ciblees
car elles reflettent le raisonnement effectif sur chaque exemple,
tandis que les regles directes sont des hypotheses globales.


### Interpretation : Extraction de regles depuis CoT

| Methode | Nombre de regles | Verdict oracle (section 3) | Source |
|---------|-----------------|----------------------------|--------|
| LLM direct | 5 candidates | 2 consistantes (les negatives), 3 rejetees | Generation globale |
| CoT extraction | 4 uniques (sur 6 traces) | 2 consistantes (les negatives), 2 rejetees | Raisonnement pas-a-pas |

Les regles extraites du CoT ont deux atouts :
1. Chaque etape du raisonnement peut etre verifiee individuellement
2. Elles sont ancrees dans des exemples concrets -- la trace sur l'exemple 4
   (negatif) produit `Patrons=Full AND Hungry=No => NOT WillWait`, une regle
   que la generation directe avait manquee

Mais l'ancrage ne garantit rien : la trace sur l'exemple 2 (mal classe par le
CoT simule, cf. section 4) re-produit la regle fautive `Patrons=Some => WillWait`.
L'oracle reste indispensable.

> **Analogie EBL** : Le CoT est a l'extraction de regles ce que la preuve EBL est a la variabilisation. La trace guide la generalisation.

---

## 5. Pipeline complet : Exemples -> LLM -> Verification -> Regles

Nous integrons maintenant toutes les etapes dans un **pipeline automatise** qui va des exemples aux regles validees.

### Architecture du pipeline

```
Exemples d'entrainement
        |
        v
[1. Prompt Engineering]  -- Construction du prompt structure
        |
        v
[2. LLM Generation]      -- Generation de regles candidates
        |
        v
[3. Parsing]             -- Extraction des clauses de Horn
        |
        v
[4. Verification]        -- Test de consistance sur les donnees
        |
        v
[5. Selection]           -- Regles validees + score de confiance
```

In [10]:
# Pipeline complet Neuro-Symbolique

@dataclass
class PipelineResult:
    """Resultat du pipeline complet."""
    n_candidates: int
    n_validated: int
    n_rejected: int
    validated_rules: list[tuple[HornClause, VerificationResult]]
    rejected_rules: list[tuple[HornClause, VerificationResult]]
    coverage: float
    

class NeuroSymbolicPipeline:
    """Pipeline complet : exemples -> LLM -> verification -> regles."""
    
    def __init__(self, use_cot: bool = False):
        self.use_cot = use_cot
        self.history: list[PipelineResult] = []
    
    def run(
        self,
        examples: list[dict],
        target: str = "WillWait",
        verbose: bool = True
    ) -> PipelineResult:
        """Execute le pipeline complet."""
        
        # Etape 1 : Construction du prompt
        prompt = build_rule_prompt(examples, target)
        
        # Etape 2 : Generation LLM (simulee)
        candidates = simulate_llm_response(prompt)
        
        # Etape 2b (optionnel) : Generation CoT
        if self.use_cot:
            # CoT sur un mix de positifs ET de negatifs : les traces sur les
            # negatifs produisent des regles negatives que la generation
            # directe peut manquer. Deduplication contre les candidates
            # directes pour ne garder que l'apport reel du CoT.
            positives = [e for e in examples if e[target]]
            negatives = [e for e in examples if not e[target]]
            seen = {str(r) for r in candidates}
            for i, ex in enumerate(positives[:3] + negatives[:3]):
                trace = simulate_cot_classification(ex, i)
                rule = trace.extracted_rule
                if rule and str(rule) not in seen:
                    seen.add(str(rule))
                    candidates.append(rule)
        
        # Etape 3 : Verification
        verified = []
        for i, rule in enumerate(candidates):
            vr = verify_rule(rule, examples, i)
            verified.append((rule, vr))
        
        # Etape 4 : Selection
        validated = [(r, v) for r, v in verified if v.is_consistent]
        rejected = [(r, v) for r, v in verified if not v.is_consistent]
        
        # Etape 5 : Calcul de la couverture
        positive_indices = {i for i, e in enumerate(examples) if e[target]}
        covered = set()
        for rule, vr in validated:
            if not rule.negated:
                for i, ex in enumerate(examples):
                    if ex[target] and rule.evaluate(ex):
                        covered.add(i)
        
        coverage = len(covered) / len(positive_indices) if positive_indices else 0.0
        
        result = PipelineResult(
            n_candidates=len(candidates),
            n_validated=len(validated),
            n_rejected=len(rejected),
            validated_rules=validated,
            rejected_rules=rejected,
            coverage=coverage
        )
        self.history.append(result)
        
        if verbose:
            self._print_result(result)
        
        return result
    
    def _print_result(self, result: PipelineResult):
        print("Pipeline Neuro-Symbolique - Resultat")
        print("=" * 55)
        print(f"  Candidates generees : {result.n_candidates}")
        print(f"  Regles validees     : {result.n_validated}")
        print(f"  Regles rejetees     : {result.n_rejected}")
        print(f"  Couverture positifs : {result.coverage:.0%}")
        print()
        print("Regles validees :")
        for rule, vr in result.validated_rules:
            print(f"  [OK] {rule}  (TP={vr.tp}, FP={vr.fp})")
        print()
        if result.rejected_rules:
            print("Regles rejetees :")
            for rule, vr in result.rejected_rules:
                print(f"  [KO] {rule}  (FP={vr.fp})")


# Execution du pipeline
pipeline = NeuroSymbolicPipeline(use_cot=False)
result = pipeline.run(EXAMPLES)

Pipeline Neuro-Symbolique - Resultat
  Candidates generees : 5
  Regles validees     : 2
  Regles rejetees     : 3
  Couverture positifs : 0%

Regles validees :
  [OK] Patrons=None => NOT WillWait  (TP=2, FP=0)
  [OK] Patrons=Full AND Price=$$$ => NOT WillWait  (TP=2, FP=0)

Regles rejetees :
  [KO] Patrons=Some => WillWait  (FP=1)
  [KO] Patrons=Full AND Hungry=Yes => WillWait  (FP=1)
  [KO] Hungry=Yes => WillWait  (FP=1)


### Execution du pipeline avec Chain-of-Thought

Le pipeline `NeuroSymbolicPipeline` ci-dessus a ete execute **sans** l'option
Chain-of-Thought. Activons maintenant le CoT pour observer ce que les traces de
raisonnement apportent au pool de candidats :

- **Sans CoT** : seules les 5 regles simulees par `simulate_llm_response()` sont candidates
- **Avec CoT** : des traces sont generees sur 3 exemples positifs et 3 negatifs ;
  apres deduplication contre les candidates directes, une seule regle nouvelle
  emerge -- `Patrons=Full AND Hungry=No => NOT WillWait`, issue d'une trace sur
  un exemple **negatif**

C'est le point interessant : la generation directe, focalisee sur "expliquer les
positifs", avait manque cette regle negative. Le raisonnement exemple-par-exemple
du CoT la fait apparaitre -- et l'oracle la valide.


In [11]:
# Execution avec Chain-of-Thought active
print("\n")
pipeline_cot = NeuroSymbolicPipeline(use_cot=True)
result_cot = pipeline_cot.run(EXAMPLES)

print(f"\nComparaison : sans CoT ({result.n_validated} validees) "
      f"vs avec CoT ({result_cot.n_validated} validees)")



Pipeline Neuro-Symbolique - Resultat
  Candidates generees : 6
  Regles validees     : 3
  Regles rejetees     : 3
  Couverture positifs : 0%

Regles validees :
  [OK] Patrons=None => NOT WillWait  (TP=2, FP=0)
  [OK] Patrons=Full AND Price=$$$ => NOT WillWait  (TP=2, FP=0)
  [OK] Patrons=Full AND Hungry=No => NOT WillWait  (TP=2, FP=0)

Regles rejetees :
  [KO] Patrons=Some => WillWait  (FP=1)
  [KO] Patrons=Full AND Hungry=Yes => WillWait  (FP=1)
  [KO] Hungry=Yes => WillWait  (FP=1)

Comparaison : sans CoT (2 validees) vs avec CoT (3 validees)


### Interpretation : Pipeline complet

Le pipeline automatise le cycle generate-and-test :

| Etape | Entree | Sortie | Verificateur |
|-------|--------|--------|-------------|
| 1. Prompt | Exemples | Prompt structure | - |
| 2. LLM | Prompt | Regles candidates | - |
| 3. Parsing | Texte LLM | HornClauses | Syntaxe |
| 4. Verification | HornClauses + donnees | Resultats | Oracle symbolique |
| 5. Selection | Resultats | Regles validees | Seuils de qualite |

**Avec CoT** : la regle supplementaire issue d'une trace sur un negatif est validee
par l'oracle (3 validees contre 2 sans CoT). La couverture des positifs reste a 0% :
toutes les regles validees sont negatives, consequence du concept disjonctif. C'est
le pipeline iteratif (section 6), avec sa tolerance aux faux positifs, qui couvrira
les positifs.

> **Point cle** : Le pipeline est **iterable**. Si la couverture est insuffisante, on peut re-prompter le LLM avec les erreurs residuelles pour generer de nouvelles hypotheses.

In [12]:
# Analyse quantitative des resultats

print("Analyse quantitative du pipeline")
print("=" * 55)
print()

for run_name, res in [("Sans CoT", result), ("Avec CoT", result_cot)]:
    print(f"--- {run_name} ---")
    print(f"  Taux d'acceptation : {res.n_validated}/{res.n_candidates} "
          f"({res.n_validated/res.n_candidates:.0%})")
    print(f"  Couverture         : {res.coverage:.0%} des positifs")
    print(f"  Regles positives   : {sum(1 for r, _ in res.validated_rules if not r.negated)}")
    print(f"  Regles negatives   : {sum(1 for r, _ in res.validated_rules if r.negated)}")
    print()

print("Metriques par regle validee (sans CoT) :")
print(f"  {'Regle':35s} | {'Precision':>9} | {'Recall':>6} | {'F1':>4}")
print("  " + "-" * 62)
for rule, vr in result.validated_rules:
    print(f"  {str(rule):35s} | {vr.precision:>9.2f} | {vr.recall:>6.2f} | {vr.f1:>4.2f}")

Analyse quantitative du pipeline

--- Sans CoT ---
  Taux d'acceptation : 2/5 (40%)
  Couverture         : 0% des positifs
  Regles positives   : 0
  Regles negatives   : 2

--- Avec CoT ---
  Taux d'acceptation : 3/6 (50%)
  Couverture         : 0% des positifs
  Regles positives   : 0
  Regles negatives   : 3

Metriques par regle validee (sans CoT) :
  Regle                               | Precision | Recall |   F1
  --------------------------------------------------------------
  Patrons=None => NOT WillWait        |      1.00 |   0.33 | 0.50
  Patrons=Full AND Price=$$$ => NOT WillWait |      1.00 |   0.33 | 0.50


### Interpretation : Metriques quantitatives

| Metrique | Signification | Valeur ideale |
|----------|--------------|---------------|
| **Taux d'acceptation** | Proportion de regles validees | Elevee = LLM competent |
| **Couverture** | Proportion de positifs couverts | 100% = modele complet |
| **Precision** | Pas de faux positifs | 1.0 = regle correcte |
| **Recall** | Couverture des positifs | Plus haut = plus generale |
| **F1** | Equilibre precision/recall | Plus haut = meilleur compromis |

> **Note** : Les regles positives (non niees) visent a couvrir les positifs. Les regles negatives (niees) visent a exclure les negatifs. Un bon modele combine les deux types.

---

## 6. Iteration et raffinement du pipeline

Le pipeline precedent est **a un seul passage**. En pratique, on peut l'ameliorer iterativement en :

1. **Analysant les erreurs** : quels exemples ne sont pas couverts ?
2. **Re-promptant** avec les erreurs residuelles pour generer de nouvelles hypotheses
3. **Raffinant** les regles existantes (specialisation si trop generales)

Ce processus est analogue au **CBH** de SL-1 : on ajuste incrementalement les hypotheses.

In [13]:
# Pipeline iteratif avec raffinement

def run_iterative_pipeline(
    examples: list[dict],
    max_fp: int = 1,
    max_iterations: int = 3,
    verbose: bool = True
) -> list[tuple[HornClause, VerificationResult]]:
    # Pipeline iteratif : genere, verifie, raffine.
    # Accepte les regles avec au plus max_fp faux positifs
    # (utile pour les concepts disjonctifs).

    all_validated = []
    uncovered_positives = [e for e in examples if e["WillWait"]]
    iteration = 0

    while uncovered_positives and iteration < max_iterations:
        iteration += 1
        if verbose:
            print(f"\nIteration {iteration} : {len(uncovered_positives)} positifs non couverts")

        # Simuler un prompt cible sur les exemples non couverts
        # (En production : envoyer un prompt avec les erreurs residuelles)
        new_candidates = simulate_llm_response("placeholder")

        # Ajouter des regles supplementaires (simulees)
        if iteration > 1:
            # Le LLM genere des regles plus ciblees en voyant les erreurs
            new_candidates.append(HornClause(
                conditions=["Patrons=Full", "Fri/Sat=Yes"],
                conclusion="WillWait"
            ))

        # Verification avec tolerance
        newly_validated = []
        for rule in new_candidates:
            vr = verify_rule(rule, examples, 0)
            # Accepter les regles avec peu de FPs et des TP > 0
            if not rule.negated and vr.fp <= max_fp and vr.tp > 0:
                if str(rule) not in {str(r) for r, _ in all_validated}:
                    newly_validated.append((rule, vr))
                    all_validated.append((rule, vr))

        if verbose:
            print(f"  Nouvelles regles validees : {len(newly_validated)}")
            for r, vr in newly_validated:
                print(f"    {r}  (TP={vr.tp}, FP={vr.fp})")

        # Mettre a jour les non-couverts
        still_uncovered = []
        for ex in uncovered_positives:
            covered = any(r.evaluate(ex) for r, _ in all_validated)
            if not covered:
                still_uncovered.append(ex)
        uncovered_positives = still_uncovered

        if verbose:
            print(f"  Positifs restants : {len(uncovered_positives)}")

    return all_validated


# Execution du pipeline iteratif
print("Pipeline iteratif avec raffinement")
print("=" * 55)
print("(Tolerance : max 1 faux positif par regle)")

final_rules = run_iterative_pipeline(EXAMPLES, max_fp=1, max_iterations=3)

print(f"\nRegles finales : {len(final_rules)}")
for i, (rule, vr) in enumerate(final_rules):
    print(f"  F{i+1}: {rule}  (TP={vr.tp}, FP={vr.fp})")

# Verifier la couverture finale
total_covered = sum(
    1 for ex in POSITIVES
    if any(r.evaluate(ex) for r, _ in final_rules)
)
print(f"\nCouverture finale : {total_covered}/{len(POSITIVES)} positifs")

# Compter les FPs totaux (union des couvertures)
all_fp_indices = set()
for rule, vr in final_rules:
    for j, ex in enumerate(EXAMPLES):
        if not ex["WillWait"] and rule.evaluate(ex):
            all_fp_indices.add(j)
print(f"Faux positifs totaux : {len(all_fp_indices)} (indices: {sorted(all_fp_indices)})")

Pipeline iteratif avec raffinement
(Tolerance : max 1 faux positif par regle)

Iteration 1 : 6 positifs non couverts
  Nouvelles regles validees : 3
    Patrons=Some => WillWait  (TP=3, FP=1)
    Patrons=Full AND Hungry=Yes => WillWait  (TP=3, FP=1)
    Hungry=Yes => WillWait  (TP=6, FP=1)
  Positifs restants : 0

Regles finales : 3
  F1: Patrons=Some => WillWait  (TP=3, FP=1)
  F2: Patrons=Full AND Hungry=Yes => WillWait  (TP=3, FP=1)
  F3: Hungry=Yes => WillWait  (TP=6, FP=1)

Couverture finale : 6/6 positifs
Faux positifs totaux : 2 (indices: [2, 9])


### Interpretation : Pipeline iteratif

Le pipeline iteratif ajoute progressivement des regles jusqu'a couverture complete :

| Iteration | Regles ajoutees | Positifs restants | Strategie |
|-----------|----------------|-------------------|----------|
| 1 | Regles initiales du LLM | Variable | Generation globale |
| 2 | Regles ciblees (si erreurs) | Reduit | Re-prompting sur erreurs |
| 3 | Regles residuelles | Minimal | Raffinement final |

**Note sur la tolerance** : Comme le concept AIMA est disjonctif, nous tolerons 1 FP par regle.
L'oracle strict (section 3) n'accepte que les regles negatives.
Le pipeline iteratif accepte aussi les regles positives "presque correctes" pour maximiser la couverture.

> **Analogie CBH** : Le pipeline iteratif est une version "amelioree" du CBH de SL-1. Au lieu d'ajuster une seule hypothese, il genere de nouvelles hypotheses a chaque iteration via le LLM.

---

## 7. Integration d'un vrai LLM

Tout ce qui precede repose sur un simulateur deterministe : ideal pour la pedagogie
(resultats reproductibles), mais l'interet pratique du paradigme est evidemment de
brancher un **vrai modele**. Cette section le fait, via n'importe quel endpoint
**OpenAI-compatible** :

| Option | Exemple | `OPENAI_BASE_URL` |
|--------|---------|-------------------|
| Cloud multi-modeles | OpenRouter (Gemini, Claude, GPT, Llama...) | `https://openrouter.ai/api/v1` |
| Cloud direct | OpenAI | `https://api.openai.com/v1` |
| Auto-heberge | vLLM / Ollama sur votre machine | `http://localhost:5002/v1` |

**Configuration** : copier le fichier [`.env.example`](./.env.example) de ce repertoire
vers `.env` et renseigner la cle API (le `.env` est gitignore, il ne sera jamais
committe). Sans `.env`, les cellules ci-dessous basculent sur le simulateur et le
notebook reste executable de bout en bout.

Le point pedagogique central ne change pas : le LLM reel est un generateur **faillible
et non deterministe** --- d'un run a l'autre, les regles proposees varient. C'est
l'oracle symbolique (section 3), lui parfaitement deterministe, qui garantit le
resultat final.


In [14]:
# Configuration de l'acces LLM via .env (endpoint OpenAI-compatible)
import os
from dotenv import load_dotenv

load_dotenv()  # charge le .env du repertoire du notebook s'il existe

LLM_API_KEY = os.getenv("OPENAI_API_KEY")
LLM_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
LLM_MODEL = os.getenv("OPENAI_CHAT_MODEL_ID", "gpt-4o-mini")
LLM_AVAILABLE = bool(LLM_API_KEY)

if LLM_AVAILABLE:
    from openai import OpenAI
    llm_client = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)
    print("LLM reel configure :")
    print(f"  Endpoint : {LLM_BASE_URL}")
    print(f"  Modele   : {LLM_MODEL}")
    print("  (la cle API n'est jamais affichee)")
else:
    llm_client = None
    print("Pas de .env / OPENAI_API_KEY : cette section utilisera le simulateur.")
    print("Copiez .env.example vers .env pour activer l'integration reelle.")

LLM reel configure :
  Endpoint : https://openrouter.ai/api/v1
  Modele   : google/gemini-3.5-flash
  (la cle API n'est jamais affichee)


In [15]:
# Appel reel du LLM et parsing robuste de sa reponse

def parse_llm_rules(text: str) -> list[HornClause]:
    """Parse les regles 'cond1 AND cond2 => [NOT] WillWait' depuis du texte libre.

    Contrairement au simulateur, un vrai LLM enrobe ses reponses (puces,
    backticks, commentaires) : on ne garde que les lignes contenant '=>' dont
    chaque condition est un litteral Attribut=Valeur connu du domaine. C'est un
    premier filtrage purement SYNTAXIQUE -- l'oracle de la section 3 fera
    ensuite le filtrage SEMANTIQUE.
    """
    rules = []
    for line in text.splitlines():
        line = line.strip().strip("`").lstrip("-*0123456789. ").strip()
        if "=>" not in line:
            continue
        body, head = line.split("=>", 1)
        head = head.strip().rstrip(".").strip("*` ")
        negated = head.upper().startswith("NOT ")
        if negated:
            head = head[4:].strip()
        if head != "WillWait":
            continue
        conditions = [c.strip() for c in body.replace("`", "").split(" AND ")]
        if conditions and all(
            "=" in c and c.split("=", 1)[0].strip() in ATTRIBUTES for c in conditions
        ):
            rules.append(HornClause(conditions=conditions,
                                    conclusion="WillWait", negated=negated))
    return rules


def llm_generate_rules(prompt: str) -> tuple[str, list[HornClause]]:
    """Envoie le prompt de la section 2 au vrai LLM et parse la reponse.

    Bascule sur le simulateur si l'API n'est pas configuree ou echoue,
    pour que le notebook reste executable hors-ligne.
    """
    if LLM_AVAILABLE:
        try:
            response = llm_client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,  # reduit (sans annuler) la variabilite
                max_tokens=4000,  # large : les modeles 'reasoning' (Gemini 3.5) consomment
                #         des tokens de reflexion AVANT la reponse visible
            )
            raw = response.choices[0].message.content
            parsed = parse_llm_rules(raw)
            if parsed:
                return raw, parsed
            print("Reponse LLM recue mais aucune regle parsable -> simulateur.")
        except Exception as exc:
            print(f"Appel LLM echoue ({type(exc).__name__}) -> simulateur.")
    rules = simulate_llm_response(prompt)
    return "\n".join(str(r) for r in rules) + "\n(reponse simulee)", rules


# On reutilise le MEME prompt que celui construit en section 2
raw_response, real_candidates = llm_generate_rules(prompt)

print(f"Reponse brute ({LLM_MODEL if LLM_AVAILABLE else 'simulateur'}) :")
print("=" * 60)
print(raw_response[:1200])
print("=" * 60)
print(f"\nRegles parsees : {len(real_candidates)}")
for i, rule in enumerate(real_candidates):
    print(f"  R{i+1}: {rule}")

Reponse brute (google/gemini-3.5-flash) :
En tant qu'expert en apprentissage symbolique (notamment via des algorithmes d'induction de règles comme CN2 ou la conversion d'arbres de décision ID3/C4.5), j'ai analysé vos données pour en extraire les motifs les plus discriminants. 

Voici un ensemble de **5 règles logiques (clauses de Horn)** qui expliquent parfaitement et sans contradiction l'intégralité de vos exemples positifs et négatifs.

### Règles générées :

1. `Hungry=No => NOT WillWait`
2. `Patrons=None => NOT WillWait`
3. `Patrons=Full AND Price=$$$ => NOT WillWait`
4. `Patrons=Some AND Hungry=Yes => WillWait`
5. `Patrons=Full AND Hungry=Yes AND Price=$ => WillWait`

---

### Explications de la couverture des données :
* **Règle 1 & 2 (Négatives fortes) :** Si le client n'a pas faim (`Hungry=No`) ou s'il n'y a personne dans le restaurant (`Patrons=None`), le client n'attend jamais (couvre les exemples négatifs 1, 2, 3, 4, 6).
* **Règle 3 (Négative contextuelle) :** Même si le cli

In [16]:
# Verification des regles du vrai LLM par le MEME oracle symbolique

print("Verification symbolique des regles generees par le vrai LLM")
print("=" * 78)
print(f"{'Regle':48s} | {'TP':>3} | {'FP':>3} | {'Prec':>5} | {'OK':>3}")
print("-" * 78)

real_validated = []
for i, rule in enumerate(real_candidates):
    vr = verify_rule(rule, EXAMPLES, i)
    ok = "OUI" if vr.is_consistent else "NON"
    print(f"{str(rule)[:48]:48s} | {vr.tp:>3} | {vr.fp:>3} | {vr.precision:>5.2f} | {ok:>3}")
    if vr.is_consistent:
        real_validated.append(rule)

print()
print(f"Bilan LLM reel  : {len(real_validated)}/{len(real_candidates)} regles validees par l'oracle")
print(f"Bilan simulateur: {len(validated_rules)}/{len(candidate_rules)} regles validees (section 3)")
print()
print("Le generateur change (simule ou reel, deterministe ou non), l'oracle")
print("applique le meme verdict : c'est lui qui rend le pipeline fiable.")

Verification symbolique des regles generees par le vrai LLM
Regle                                            |  TP |  FP |  Prec |  OK
------------------------------------------------------------------------------
Hungry=No => NOT WillWait                        |   5 |   0 |  1.00 | OUI
Patrons=None => NOT WillWait                     |   2 |   0 |  1.00 | OUI
Patrons=Full AND Price=$$$ => NOT WillWait       |   2 |   0 |  1.00 | OUI
Patrons=Some AND Hungry=Yes => WillWait          |   3 |   0 |  1.00 | OUI
Patrons=Full AND Hungry=Yes AND Price=$ => WillW |   3 |   0 |  1.00 | OUI

Bilan LLM reel  : 5/5 regles validees par l'oracle
Bilan simulateur: 2/5 regles validees (section 3)

Le generateur change (simule ou reel, deterministe ou non), l'oracle
applique le meme verdict : c'est lui qui rend le pipeline fiable.


### Interpretation : le generateur change, l'oracle demeure

Trois observations sur cette execution reelle :

1. **Le parsing devient une vraie etape.** Le simulateur renvoyait directement des
   objets `HornClause` ; le vrai modele renvoie du texte libre qu'il faut filtrer
   syntaxiquement (lignes malformees, attributs inventes) avant meme la verification
   semantique. Dans un pipeline neuro-symbolique reel, ce parseur est un point de
   defaillance a part entiere.

2. **Non-determinisme assume.** Les sorties committees dans ce notebook correspondent
   a *un* run du modele indique ci-dessus (`temperature=0` reduit la variabilite sans
   la supprimer). En re-executant, vous obtiendrez probablement un ensemble de regles
   different --- mais le verdict de l'oracle sur chaque regle, lui, ne changera jamais.

3. **La division du travail tient --- meme quand le generateur est bon.** Sur ce
   run (Gemini 3.5 Flash, modele *reasoning* : il depense un budget de tokens en
   reflexion interne avant de repondre), les 5 regles proposees passent l'oracle
   (FP=0 partout), la ou le simulateur n'en validait que 2/5. Les deux regles
   positives se partagent exactement les 6 exemples positifs (3+3) : le modele a
   retrouve seul la structure du dilemme couverture/consistance des sections 2-3.
   Faut-il alors encore un oracle ? Oui : c'est lui qui *prouve* que ce run est bon.
   Avec un modele plus faible (essayez dans le `.env`), une partie des regles ---
   plausibles en langage naturel --- est refutee par les donnees ; sans verification
   symbolique, rien ne distingue ces deux situations.


---

## 8. Exercices

### Exemple guide 1 : Ajouter une regle via prompt personnalise (domaine film)

Nous appliquons le pipeline des sections 2-3 a un **nouveau domaine** : la prediction de la qualite d'un film. La cible n'est plus `WillWait` mais `bon_film` -- il faut donc adapter la verification (qui utilisait `ex["WillWait"]` en dur).

**Etapes** :
1. Definir les attributs du domaine (genre, duree, budget, critique)
2. Creer 6 exemples (3 positifs, 3 negatifs)
3. Generer 3 regles candidates (simulees)
4. Verifier chaque regle avec l'oracle symbolique (cible `bon_film`)

In [17]:
# Exemple guide 1 : domaine film -- generation + verification symbolique
#
# On reprend l'architecture des sections 2-3 sur un nouveau domaine.
# Concept sous-jacent (que l'oracle doit isoler) : un bon film est simplement
# bien critique, independamment du genre, de la duree ou du budget.

# Etape 1 : les 4 attributs du domaine film
FILM_ATTRIBUTES = ["genre", "duree", "budget", "critique"]

# Etape 2 : 6 exemples (3 positifs, 3 negatifs)
film_examples = [
    {"genre": "drama",  "duree": "long",  "budget": "high", "critique": "good", "bon_film": True},
    {"genre": "drama",  "duree": "court", "budget": "low",  "critique": "good", "bon_film": True},
    {"genre": "comedy", "duree": "court", "budget": "low",  "critique": "good", "bon_film": True},
    {"genre": "action", "duree": "long",  "budget": "high", "critique": "bad",  "bon_film": False},
    {"genre": "comedy", "duree": "court", "budget": "low",  "critique": "bad",  "bon_film": False},
    {"genre": "action", "duree": "court", "budget": "high", "critique": "bad",  "bon_film": False},
]

# Etape 3 : 3 regles candidates (comme ce que produirait un LLM) :
# une correcte (R1), une consistante mais incomplete (R2), une erronee (R3).
film_rules = [
    HornClause(["critique=good"], "bon_film"),   # R1
    HornClause(["genre=drama"], "bon_film"),     # R2
    HornClause(["budget=high"], "bon_film"),     # R3
]

# Etape 4 : verification. La cible change ("bon_film"), on generalise donc
# verify_rule en parametrant la cle cible -- meme logique TP/FP/FN/TN qu'en
# section 3, appliquee a un domaine quelconque.
def verify_film_rule(rule, examples, target="bon_film"):
    """verify_rule generalisee : la cle cible devient un parametre."""
    tp = fp = fn = tn = 0
    for ex in examples:
        applies = rule.evaluate(ex)
        is_pos = ex[target]
        if rule.negated:
            if applies and not is_pos:
                tp += 1
            elif applies and is_pos:
                fp += 1
            elif (not applies) and (not is_pos):
                fn += 1
            else:
                tn += 1
        else:
            if applies and is_pos:
                tp += 1
            elif applies and (not is_pos):
                fp += 1
            elif (not applies) and is_pos:
                fn += 1
            else:
                tn += 1
    consistent = (fp == 0)
    covers_all = (fn == 0) if not rule.negated else True
    return tp, fp, fn, tn, consistent, covers_all


print("Verification des regles candidates du domaine film")
print("=" * 64)
print(f"{'Regle':32s} | {'TP':>2} | {'FP':>2} | {'FN':>2} | {'OK':>3} | {'Couv.':>5}")
print("-" * 64)
for i, rule in enumerate(film_rules):
    tp, fp, fn, tn, ok, cov = verify_film_rule(rule, film_examples)
    print(f"R{i+1}: {str(rule)[:28]:28s} | {tp:>2} | {fp:>2} | {fn:>2} | "
          f"{'OUI' if ok else 'NON':>3} | {'oui' if cov else 'non':>5}")

print()
print("Lecture : R1 (critique=good) est a la fois consistante (FP=0) ET complete")
print("(FN=0) : c'est la regle cherchee. R2 est consistante mais incomplete")
print("(elle oublie la comedie positive) ; R3 est incoherente (budget=high couvre")
print("aussi des exemples negatifs). L'oracle isole donc R1, exactement comme en")
print("section 3 sur le domaine du restaurant.")

Verification des regles candidates du domaine film
Regle                            | TP | FP | FN |  OK | Couv.
----------------------------------------------------------------
R1: critique=good => bon_film    |  3 |  0 |  0 | OUI |   oui
R2: genre=drama => bon_film      |  2 |  0 |  1 | OUI |   non
R3: budget=high => bon_film      |  1 |  2 |  2 | NON |   non

Lecture : R1 (critique=good) est a la fois consistante (FP=0) ET complete
(FN=0) : c'est la regle cherchee. R2 est consistante mais incomplete
(elle oublie la comedie positive) ; R3 est incoherente (budget=high couvre
aussi des exemples negatifs). L'oracle isole donc R1, exactement comme en
section 3 sur le domaine du restaurant.


### Exercice 1 (variation) : Domaine "activite en plein air"

Reproduisez l'exemple guide ci-dessus sur un **domaine different** : decider si l'on fait une activite en plein air (`dehors`). Vous choisissez les attributs, vous construisez 6 exemples equilibres (3 ou l'on sort, 3 ou l'on reste) caches derriere un concept simple, puis vous proposez 3 regles candidates dont une seule est parfaite. Adaptez la verification a la cible `dehors`.

**Etapes** :
1. Choisir 3 ou 4 attributs et leurs valeurs (ex : `meteo`, `vent`, `temperature`, `weekend`)
2. Construire 6 exemples (3 `dehors=True`, 3 `dehors=False`) caches derriere un concept mono-attribut
3. Proposer 3 `HornClause` : 1 parfaite, 1 consistante mais incomplete, 1 incoherente
4. Verifier avec une cible parametree (`dehors`) et afficher le tableau TP/FP/FN/OK/Couverture

# Indice : un concept mono-attribut (un seul predicat decisif, ex `meteo=soleil`)
#           est le plus simple a isoler -- comme `critique=good` dans l'exemple.

In [18]:
# Exercice 1 (variation) : domaine "activite en plein air"
# TODO etudiant : definissez le domaine, 6 exemples, 3 candidats, puis verifiez.

# Etape 1 : attributs
OUTDOOR_ATTRIBUTES = []  # TODO etudiant : ex ["meteo", "vent", "temperature", "weekend"]

# Etape 2 : 6 exemples (3 dehors=True, 3 dehors=False)
outdoor_examples = [
    # TODO etudiant : ex {"meteo": "soleil", "vent": "non", "temperature": "douce",
    #                      "weekend": "oui", "dehors": True}
]

# Etape 3 : 3 regles candidates (1 parfaite, 1 incomplete, 1 incoherente)
outdoor_rules = []  # TODO etudiant : ajoutez vos HornClause

# Etape 4 : verification (adaptez verify_film_rule avec target="dehors")
# TODO etudiant : affichez le tableau TP/FP/FN/OK/Couverture comme ci-dessus.

print("Exercice a completer : domaine activite en plein air")

Exercice a completer : domaine activite en plein air


### Exemple guide 2 : Comparer prompt direct vs Chain-of-Thought (CoT)

Cet exemple guide compare les regles produites par une **generation directe** (`use_cot=False`) a celles enrichies par des **traces Chain-of-Thought** (`use_cot=True`). Les deux lots sont verifies par le meme oracle symbolique ; on compare ensuite le nombre de regles validees, le taux d'acceptation et la couverture des exemples positifs.

**Etapes** :
1. Executer le pipeline sans CoT (`verbose=False`)
2. Executer le pipeline avec CoT (`verbose=False`)
3. Comparer le nombre de regles validees, le taux d'acceptation et la couverture


In [19]:
# Exemple guide 2 : Comparaison generation directe vs Chain-of-Thought (CoT)
#
# On execute deux fois le pipeline Neuro-Symbolique sur le meme jeu
# d'exemples (EXAMPLES) : en generation directe, puis en ajoutant les
# regles issues des traces CoT. Le meme oracle verifie les deux lots.
# On compare alors n_validated, le taux d'acceptation et la couverture.

# Etape 1 : Pipeline sans CoT (generation directe)
p_direct = NeuroSymbolicPipeline(use_cot=False)
r_direct = p_direct.run(EXAMPLES, verbose=False)

# Etape 2 : Pipeline avec CoT (traces sur positifs ET negatifs)
p_cot = NeuroSymbolicPipeline(use_cot=True)
r_cot = p_cot.run(EXAMPLES, verbose=False)

# Etape 3 : Taux d'acceptation = regles valides / candidates generees
def taux_acceptation(result):
    return result.n_validated / result.n_candidates if result.n_candidates else 0.0

rows = [
    ("Candidates generees", r_direct.n_candidates, r_cot.n_candidates, "d"),
    ("Regles validees (consistantes)", r_direct.n_validated, r_cot.n_validated, "d"),
    ("Regles rejetees (inconsistantes)", r_direct.n_rejected, r_cot.n_rejected, "d"),
    ("Taux d'acceptation", taux_acceptation(r_direct), taux_acceptation(r_cot), ".0%"),
    ("Couverture positifs", r_direct.coverage, r_cot.coverage, ".0%"),
]

print("Comparaison generation directe vs CoT")
print("=" * 64)
print(f"{'Metrique':34s} | {'Direct':>12s} | {'CoT':>12s}")
print("-" * 64)
for label, vd, vc, fmt in rows:
    print(f"{label:34s} | {format(vd, fmt):>12s} | {format(vc, fmt):>12s}")
print()
print("Lecture : sur CE jeu d'exemples, aucune regle POSITIVE consistante")
print("n'existe (toutes ont FP>=1) : le concept positif est disjonctif. Les")
print("regles validees sont donc toutes NEGATIVES (NOT WillWait). Le CoT, en")
print("generant a partir des traces sur positifs ET negatifs, decouvre une")
print("regle negative supplementaire (3 validees vs 2 en direct), d'ou un")
print("taux d'acceptation plus eleve (50% vs 40%). La couverture des positifs")
print("reste 0% dans les deux cas : c'est une limite du formalisme Horn sur un")
print("concept disjonctif, pas une defaillance du CoT.")


Comparaison generation directe vs CoT
Metrique                           |       Direct |          CoT
----------------------------------------------------------------
Candidates generees                |            5 |            6
Regles validees (consistantes)     |            2 |            3
Regles rejetees (inconsistantes)   |            3 |            3
Taux d'acceptation                 |          40% |          50%
Couverture positifs                |           0% |           0%

Lecture : sur CE jeu d'exemples, aucune regle POSITIVE consistante
n'existe (toutes ont FP>=1) : le concept positif est disjonctif. Les
regles validees sont donc toutes NEGATIVES (NOT WillWait). Le CoT, en
generant a partir des traces sur positifs ET negatifs, decouvre une
regle negative supplementaire (3 validees vs 2 en direct), d'ou un
taux d'acceptation plus eleve (50% vs 40%). La couverture des positifs
reste 0% dans les deux cas : c'est une limite du formalisme Horn sur un
concept disjonctif, pa

### Exercice 2 (variation) : Effet du nombre d'exemples sur la couverture

Au lieu de comparer deux modes de generation (direct vs CoT), comparez le comportement du pipeline quand on lui donne **un nombre croissant d'exemples**. On s'attend a ce que la couverture des positifs augmente avec le nombre d'exemples, jusqu'a un plateau une fois toutes les sous-classes couvertes.

**Etapes** :
1. Construire des sous-ensembles d'EXAMPLES de tailles 3, 5, puis la taille complete
2. Executer le pipeline (generation directe) sur chaque sous-ensemble
3. Afficher l'evolution de `n_validated` et de `coverage` selon la taille



In [20]:
# Exercice 2 (variation) : effet du nombre d'exemples sur la couverture
# TODO etudiant : comparez le pipeline sur des sous-ensembles croissants

# Etape 1 : sous-ensembles de tailles 3, 5, puis len(EXAMPLES)
# tailles = [3, 5, len(EXAMPLES)]
# sous_ensembles = {k: EXAMPLES[:k] for k in tailles}

# Etape 2 : pipeline en generation directe sur chaque sous-ensemble
# resultats = []
# for k, exs in sous_ensembles.items():
#     p = NeuroSymbolicPipeline(use_cot=False)
#     r = p.run(exs, verbose=False)
#     resultats.append((k, r.n_validated, r.coverage))

# Etape 3 : afficher l'evolution de n_validated et coverage selon la taille
print("Exercice a completer : effet du nombre d'exemples sur la couverture")
print("Etape 1 : construisez des sous-ensembles de tailles 3, 5, len(EXAMPLES)")
print("Etape 2 : executez le pipeline (direct) sur chaque sous-ensemble")
print("Etape 3 : affichez l'evolution de n_validated et coverage")

# Indice : la couverture des positifs augmente avec le nombre d'exemples,
# puis se stabilise (plateau) une fois toutes les sous-classes couvertes.
# result = None  # TODO etudiant : liste de tuples (taille, n_validated, coverage)


Exercice a completer : effet du nombre d'exemples sur la couverture
Etape 1 : construisez des sous-ensembles de tailles 3, 5, len(EXAMPLES)
Etape 2 : executez le pipeline (direct) sur chaque sous-ensemble
Etape 3 : affichez l'evolution de n_validated et coverage


### Exemple guide 3 : Detecter les hallucinations du LLM

Un LLM peut generer des regles qui semblent correctes mais qui sont en fait des **hallucinations** : conditions impossibles (contradiction sur un meme attribut), attributs hors domaine, ou tautologie (regle sans condition, qui conclut toujours). Cet exemple guide implemente un detecteur qui parcourt les conditions d'une regle et signale ces trois familles de problemes.

**Etapes** :
1. Definir ce qu'est une regle "hallucinee" (contradiction, attribut invalide, tautologie)
2. Implementer `detect_hallucination()`
3. Tester sur des regles problematiques (contradiction, attribut inexistant, tautologie)


In [21]:
# Exemple guide 3 : Detection d'hallucinations


def detect_hallucination(rule: HornClause, valid_attrs: list[str], valid_values: dict) -> list[str]:
    """Detecte les problemes potentiels dans une regle.

    Retourne une liste de problemes detectes (vide = OK).
    Trois familles d'hallucinations :
      - Tautologie : aucune condition (la regle conclut toujours, sans specialisation).
      - Attribut invalide : une condition porte sur un attribut inconnu,
        ou une valeur hors du domaine autorise.
      - Contradiction : deux conditions sur le MEME attribut avec des
        valeurs differentes (la regle n'est jamais satisfaite).
    """
    problems = []

    # Tautologie : aucune condition -> la regle conclut dans tous les cas
    if not rule.conditions:
        problems.append("Tautologie : la regle n'a aucune condition "
                        "(elle conclut toujours, sans specialisation)")
        return problems  # rien d'autre a verifier

    seen: dict[str, str] = {}  # attribut -> premiere valeur vue
    for cond in rule.conditions:
        if "=" not in cond:
            problems.append(f"Condition mal formee (sans '=') : '{cond}'")
            continue
        attr, _, value = cond.partition("=")
        attr, value = attr.strip(), value.strip()

        # Attribut invalide : hors du domaine d'attributs connus
        if attr not in valid_attrs:
            problems.append(f"Attribut invalide : '{attr}' (hors du domaine connu)")
            continue  # inutile de verifier la valeur d'un attribut inconnu

        # Valeur hors domaine
        if attr in valid_values and value not in valid_values[attr]:
            problems.append(f"Valeur hors domaine : '{attr}={value}' "
                            f"(valeurs valides : {valid_values[attr]})")

        # Contradiction : meme attribut deja vu avec une valeur differente
        if attr in seen and seen[attr] != value:
            problems.append(f"Contradiction : '{attr}={seen[attr]}' et "
                            f"'{attr}={value}' sont incompatibles")
        else:
            seen[attr] = value

    return problems


# Tests : regles problematiques a detecter
test_rules = [
    HornClause(["Patrons=Full", "Patrons=Some"], "WillWait"),   # Contradiction
    HornClause(["AttributInexistant=v"], "WillWait"),            # Attribut invalide
    HornClause([], "WillWait"),                                   # Tautologie (pas de conditions)
]

valid_values = {
    "Patrons": ["Some", "Full", "None"],
    "Hungry": ["Yes", "No"],
    "Type": ["French", "Thai", "Burger", "Italian"],
}
valid_attrs = list(valid_values.keys())

print("Detection d'hallucinations sur les regles problematiques")
print("=" * 62)
for i, rule in enumerate(test_rules):
    problems = detect_hallucination(rule, valid_attrs, valid_values)
    statut = "OK (pas d'hallucination)" if not problems else "HALLUCINATION"
    print(f"H{i+1}: {rule}")
    print(f"    -> {statut}")
    for p in problems:
        print(f"       - {p}")
    print()

print("Lecture : les trois regles declenchent chacune une famille distincte")
print("d'hallucination. H1 est une contradiction (Patrons ne peut valoir Full")
print("ET Some a la fois) ; H2 cite un attribut inconnu du domaine ; H3 est une")
print("tautologie (sans condition, la regle conclut pour TOUT exemple, donc ne")
print("specialise rien). Un oracle symbolique seul ne detecte pas ces defauts")
print("(il mesure TP/FP/FN) : ce detecteur complementaire filtre les candidats")
print("du LLM AVANT la verification, pour eviter de valider des regles vides")
print("de sens.")


Detection d'hallucinations sur les regles problematiques
H1: Patrons=Full AND Patrons=Some => WillWait
    -> HALLUCINATION
       - Contradiction : 'Patrons=Full' et 'Patrons=Some' sont incompatibles

H2: AttributInexistant=v => WillWait
    -> HALLUCINATION
       - Attribut invalide : 'AttributInexistant' (hors du domaine connu)

H3:  => WillWait
    -> HALLUCINATION
       - Tautologie : la regle n'a aucune condition (elle conclut toujours, sans specialisation)

Lecture : les trois regles declenchent chacune une famille distincte
d'hallucination. H1 est une contradiction (Patrons ne peut valoir Full
ET Some a la fois) ; H2 cite un attribut inconnu du domaine ; H3 est une
tautologie (sans condition, la regle conclut pour TOUT exemple, donc ne
specialise rien). Un oracle symbolique seul ne detecte pas ces defauts
(il mesure TP/FP/FN) : ce detecteur complementaire filtre les candidats
du LLM AVANT la verification, pour eviter de valider des regles vides
de sens.


### Exercice 3 (variation) : Detecter une regle non confirmee par les donnees

Une regle peut etre **formellement valide** (pas de contradiction, attributs corrects, conditions presentes) et pourtant constituer une **hallucination vis-a-vis des donnees** : si AUCUN exemple d'entrainement ne satisfait toutes ses conditions, la regle n'est jamais declenchee et n'est pas confirmee par l'experience. Etendez le detecteur pour signaler ces regles "mortes".

**Etapes** :
1. Ecrire `regle_non_confirmee(rule, examples)` : retourne `True` si aucun exemple ne satisfait toutes les conditions de la regle
2. Appliquer `detect_hallucination` (formel) PUIS `regle_non_confirmee` (donnees) sur un jeu de regles
3. Distinguer les deux types de problemes dans l'affichage



In [22]:
# Exercice 3 (variation) : regle non confirmee par les donnees
# TODO etudiant : une regle formellement valide peut etre "morte" si
# aucun exemple d'entrainement ne satisfait toutes ses conditions.

# Etape 1 : fonction de couverture par les donnees
def regle_non_confirmee(rule, examples):
    """Retourne True si AUCUN exemple ne satisfait toutes les conditions."""
    # TODO etudiant : parcourez examples, testez rule.evaluate(ex)
    return None  # TODO etudiant

# Etape 2 : combiner detection formelle + couverture donnees
# for rule in regles_test:
#     formel = detect_hallucination(rule, valid_attrs, valid_values)
#     morte = regle_non_confirmee(rule, EXAMPLES)
#     ...

# Etape 3 : afficher les deux types de problemes distinctement
print("Exercice a completer : detecter les regles non confirmees par les donnees")
print("Etape 1 : ecrivez regle_non_confirmee(rule, examples)")
print("Etape 2 : combinez detect_hallucination (formel) et regle_non_confirmee (donnees)")
print("Etape 3 : affichez les deux types de problemes distinctement")

# Indice : HornClause.evaluate(ex) retourne True si toutes les conditions
# sont satisfaites ; une regle "morte" n'a aucun exemple qui l'active.
# result = None  # TODO etudiant


Exercice a completer : detecter les regles non confirmees par les donnees
Etape 1 : ecrivez regle_non_confirmee(rule, examples)
Etape 2 : combinez detect_hallucination (formel) et regle_non_confirmee (donnees)
Etape 3 : affichez les deux types de problemes distinctement


### Exemple guide 4 : Mesurer le taux d'hallucination du generateur

La section 7 a valide **UN** lot de regles du vrai LLM. Un seul tirage ne mesure rien : la generation est stochastique. Cet exemple guide rejoue plusieurs tirages et mesure quelle fraction des regles produites survit a l'oracle symbolique, puis comment ce taux evolve avec le "bruit" de la generation.

Sans `.env`, on substitue la stochasticite de la **graine** a celle de la temperature (cf. indice) : la question posee -- *l'oracle stabilise-t-il le pipeline face a la stochasticite de la generation ?* -- est identique.

**Etapes** :
1. Ecrire `generate_rules(seed, bruit)` qui echantillonne un bassin de candidats selon la graine (generation "focalisee" vs "bruitee")
2. Pour chaque regime, faire 5 tirages ; pour chaque tirage, compter `n_generees` et `n_validees` (`verify_rule` sur `EXAMPLES`, garder `is_consistent`)
3. Afficher par regime : total genere, total valide, taux de validation
4. L'oracle rend-il le pipeline insensible au bruit de generation ? Qu'est-ce qui change vraiment (cout, diversite) ?


In [23]:
# Exemple guide 4 : Taux d'hallucination du generateur (l'oracle comme filtre)
#
# La section 7 a valide UN seul tirage du LLM. Un seul tirage ne mesure
# rien : la generation est stochastique. Ici on rejoue plusieurs tirages
# et on mesure quelle fraction des regles produites survit a l'oracle
# symbolique, et comment ce taux evolve avec le "bruit" de generation.
#
# Sans .env (pas de vrai LLM), on substitue la stochasticite de la GRAINE
# a celle de la temperature (cf. indice) : la question posee est identique.

import random

# Bassin de candidats : melange de regles CONSISTANTES (validables par
# l'oracle) et INCONSISTANTES (rejetees). Leur statut est connu grace a
# la section 3 (cf. cellule 8917b738 sur origin/main).
CONSISTANTES = [
    HornClause(["Patrons=None"], "WillWait", negated=True),
    HornClause(["Patrons=Full", "Price=$$$"], "WillWait", negated=True),
    HornClause(["Patrons=Full", "Hungry=No"], "WillWait", negated=True),
]
INCONSISTANTES = [
    HornClause(["Patrons=Some"], "WillWait"),
    HornClause(["Patrons=Full", "Hungry=Yes"], "WillWait"),
    HornClause(["Hungry=Yes"], "WillWait"),
    HornClause(["Type=French"], "WillWait"),
]


def generate_rules(seed: int, bruit: bool):
    """Simule un tirage LLM stochastique.

    bruit=False -> generation 'focalisee' (temperature basse) : pioche
                   dans un bassin a dominante consistante.
    bruit=True  -> generation 'bruitee' (temperature elevee) : pioche
                   dans le bassin complet, donc plus d'hallucinations.
    """
    pool = CONSISTANTES + (INCONSISTANTES if bruit else INCONSISTANTES[:1])
    k = 6 if bruit else 4
    return random.Random(seed).sample(pool, k)


def stats_tirage(rules):
    """Retourne (n_generees, n_validees) selon l'oracle sur EXAMPLES."""
    n_val = sum(
        1 for i, r in enumerate(rules)
        if verify_rule(r, EXAMPLES, i).is_consistent
    )
    return len(rules), n_val


regimes = [("temperature ~0.2 (focalise)", False, 20),
           ("temperature ~0.9 (bruite)", True, 90)]

print("Taux de validation oracle par regime de generation (simulateur, 5 tirages)")
print("=" * 70)
print(f"{'Regime':32s} | {'Generees':>9s} | {'Validees':>9s} | {'Taux':>6s}")
print("-" * 70)
for label, bruit, base_seed in regimes:
    tot_gen = tot_val = 0
    for draw in range(5):
        rules = generate_rules(base_seed + draw, bruit)
        g, v = stats_tirage(rules)
        tot_gen += g
        tot_val += v
    taux = tot_val / tot_gen if tot_gen else 0.0
    print(f"{label:32s} | {tot_gen:>9d} | {tot_val:>9d} | {taux:>5.0%}")
print()
print("Lecture : le regime bruite genere PLUS de regles au total (bassin plus")
print("large, k plus eleve) mais son taux de validation est plus bas (bassin")
print("dilue d'inconsistantes). L'oracle symbolique filtre les deux regimes de")
print("la meme facon : ne survivent que les regles consistantes (FP=0). La")
print("QUALITE du resultat final est donc stable ; ce qui change vraiment,")
print("c'est le COUT (plus d'appels LLM et de verification) et la DIVERSITE")
print("exploree (plus de candidats, donc plus de chances de decouvrir une")
print("regle utile, au prix de plus de bruit).")
print()
print("Conclusion : un seul tirage (section 7) n'est PAS representatif. Seul")
print("l'oracle rend le pipeline robuste a la stochasticite de la generation.")


Taux de validation oracle par regime de generation (simulateur, 5 tirages)
Regime                           |  Generees |  Validees |   Taux
----------------------------------------------------------------------
temperature ~0.2 (focalise)      |        20 |        15 |   75%
temperature ~0.9 (bruite)        |        30 |        13 |   43%

Lecture : le regime bruite genere PLUS de regles au total (bassin plus
large, k plus eleve) mais son taux de validation est plus bas (bassin
dilue d'inconsistantes). L'oracle symbolique filtre les deux regimes de
la meme facon : ne survivent que les regles consistantes (FP=0). La
QUALITE du resultat final est donc stable ; ce qui change vraiment,
c'est le COUT (plus d'appels LLM et de verification) et la DIVERSITE
exploree (plus de candidats, donc plus de chances de decouvrir une
regle utile, au prix de plus de bruit).

Conclusion : un seul tirage (section 7) n'est PAS representatif. Seul
l'oracle rend le pipeline robuste a la stochasticite de la ge

### Exercice 4 (variation) : Stabilite du taux de validation (intervalle de confiance)

Le taux de validation mesure sur 5 tirages reste une estimation bruitee. Combien de tirages faut-il pour l'estimer fiablement ? Etendez l'etude : tirez `N=20` fois a un regime fixe, calculez la **moyenne** et l'**ecart-type** du taux de validation par tirage, et donnez un intervalle de confiance approximatif (moyenne +/- ecart-type).

**Etapes** :
1. Reutiliser `generate_rules` + `stats_tirage` pour `N=20` tirages a un regime fixe
2. Calculer le taux de validation de chaque tirage, puis leur moyenne et ecart-type
3. Afficher l'intervalle moyenne +/- ecart-type et commenter la stabilite de l'estimation



In [24]:
# Exercice 4 (variation) : stabilite du taux de validation
# TODO etudiant : estimez le taux de validation avec un intervalle de confiance.

# Etape 1 : N=20 tirages a un regime fixe (reutilisez generate_rules + stats_tirage)
# taux_par_tirage = []
# for draw in range(20):
#     rules = generate_rules(1000 + draw, bruit=True)
#     g, v = stats_tirage(rules)
#     taux_par_tirage.append(v / g)

# Etape 2 : moyenne et ecart-type du taux de validation
# moyenne = sum(taux_par_tirage) / len(taux_par_tirage)
# variance = sum((t - moyenne) ** 2 for t in taux_par_tirage) / len(taux_par_tirage)
# ecart_type = variance ** 0.5

# Etape 3 : intervalle moyenne +/- ecart-type
print("Exercice a completer : stabilite du taux de validation sur N=20 tirages")
print("Etape 1 : calculez le taux de validation de chacun des 20 tirages")
print("Etape 2 : derivez moyenne et ecart-type")
print("Etape 3 : affichez l'intervalle moyenne +/- ecart-type")

# Indice : plus l'ecart-type est petit devant la moyenne, plus l'estimation
# est stable -- et moins il faut de tirages pour conclure.
# result = None  # TODO etudiant : (moyenne, ecart_type)


Exercice a completer : stabilite du taux de validation sur N=20 tirages
Etape 1 : calculez le taux de validation de chacun des 20 tirages
Etape 2 : derivez moyenne et ecart-type
Etape 3 : affichez l'intervalle moyenne +/- ecart-type


---

## 9. Conclusion

### Resume des concepts

| Concept | Description | Analogie classique |
|---------|-------------|--------------------|
| **LLM comme generateur** | Produit des hypotheses plausibles depuis des exemples | Heuristique de recherche |
| **Oracle symbolique** | Verifie formellement les hypotheses candidates | Test de consistance (CBH) |
| **Chain-of-Thought** | Raisonnement etape par etape du LLM | Arbre de preuve EBL |
| **Extraction de regles** | Parser les traces CoT en clauses logiques | Variabilisation EBL |
| **Pipeline iteratif** | Generer -> Verifier -> Raffiner | CBH avec backtracking |
| **Detection d'hallucinations** | Filtrer les regles impossibles ou triviales | Verification de type |

### Forces et limites de l'approche

| Aspect | Avantage | Limite |
|--------|----------|--------|
| **Creativite** | Le LLM genere des hypotheses variees | Peut halluciner |
| **Vitesse** | Plus rapide que la recherche exhaustive | Depend du temps d'inference LLM |
| **Fiabilite** | L'oracle garantit la correction | Ne detecte pas les omissions |
| **Generalisation** | Contexte naturel du LLM | Limite par le prompt engineering |
| **Scalabilite** | Pas besoin de domaine formel complet | Cout des appels API |

### Liens avec les notebooks precedents

| Notebook | Connexion avec SL-7 |
|----------|---------------------|
| SL-1 (CBH/VS) | Le pipeline iteratif est analogue au CBH, avec le LLM comme generateur |
| SL-2 (EBL) | Le CoT est une forme d'arbre de preuve EBL en langage naturel |
| SL-4 (ILP) | L'oracle symbolique reprend les tests de couverture de l'ILP |

### Pour aller plus loin

- Brancher un vrai LLM (OpenAI, Anthropic, ou modele local via Ollama) pour voir les differences de qualite
- Comparer avec les approches ILP pures (Progol, Aleph) sur des benchmarks classiques
- Explorer le **self-consistency** : generer N reponses et voter pour la plus frequente
- Etudier les connections avec TweetyProject pour la verification logique avancee

---

## Defi presentation

Modalite du cours : chaque groupe choisit **un exercice** de la serie, le prepare,
et le presente en seance. Resoudre l'exercice est le minimum ; ce qui distingue une
presentation qui maitrise le sujet, c'est la **question-twist** associee ci-dessous.
Elle fait partie integrante de la presentation attendue.

| Exercice | Question-twist a traiter en plus |
|----------|----------------------------------|
| **Ex. 1 (prompt personnalise)** | Changez de modele dans le `.env` (un modele plus petit via OpenRouter) et comparez le taux de validation oracle avec Gemini 3.5 Flash sur le meme prompt. Que conclure sur la dependance du pipeline au generateur --- et sur ce que l'oracle garantit vraiment ? |
| **Ex. 2 (prompt direct vs CoT)** | Le CoT ameliore-t-il le taux de validation *oracle* ou seulement la plausibilite du texte ? Les deux metriques peuvent diverger : montrez (ou construisez) un cas. |
| **Ex. 3 (detection d'hallucinations)** | Votre detecteur s'appuie sur un oracle a donnees completes (monde clos). Que devient-il si les donnees sont incompletes (monde ouvert, cf SL-6) ? Proposez et justifiez une adaptation. |
| **Ex. 4 (taux d'hallucination)** | Le taux d'hallucination est-il un defaut du generateur ou un parametre d'exploration ? Cherchez le reglage qui maximise le nombre de regles VALIDEES par appel (et non le taux de validation) : pourquoi ces deux objectifs divergent-ils, et lequel l'architecture generateur+oracle permet-elle d'assumer sans risque ? |


---

## Ressources

- Russell & Norvig, *AI: A Modern Approach*, 4e ed., Chapitre 19
- Wei et al., *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* (NeurIPS 2022)
- Muggleton, *Inductive Logic Programming* (1991)
- Pan et al., *Unifying Large Language Models and Knowledge Graphs: A Roadmap* (2024)
- [TweetyProject](https://tweetyproject.org/) - Bibliotheque Java pour la logique computationnelle

---

**Retour** : [Index SymbolicLearning](./README.md) | [<< Knowledge Graphs ILP](SL-6-KnowledgeGraphs-ILP.ipynb) | [SL-8 >>](SL-8-ActiveAutomataLearning.ipynb)